# 05 · The default model: full run R01 and a look at what it learned (tasks T17 and T18)

R01 is the main model and the reference that every ablation is compared with:
MobileNetV2 encoder (random initial weights), VV + VH + ratio input, BCE + Dice loss, seed 0.

* **T17**: train for up to 60 epochs with learning-rate reduction and early stopping on the validation water IoU, and keep the best-validation model.
* **T18**: inspect the training curves and the predictions **on validation chips only**, and write down what they show, before the training defaults are frozen.

**Test and Bolivia are not opened in this notebook.** R01 is scored on them only after the defaults are frozen (task T19), in notebook 06, so that no test number exists while the method can still change.

In [ ]:
import json
import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "config.py").exists())
sys.path.insert(0, str(ROOT))
import src  # noqa: F401  (must come before TensorFlow: switches on the XLA compile cache)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src import viz
from src.config import load_config, run_dir
from src.runner import run_training

viz.use_style()
pd.set_option("display.float_format", lambda v: f"{v:.4f}")
RUN = "R01"
cfg = load_config(RUN)
pd.Series(cfg, name=RUN).to_frame()

## 1. Train (T17)

Callbacks, all watching the **validation water IoU**: keep the best model, halve the learning rate after 5 epochs without improvement, stop after 15 epochs without improvement.

In [ ]:
metrics = run_training(RUN, evaluate=False)

In [ ]:
t = metrics["training"]
pd.Series({
    "epochs trained (of at most 60)": t["epochs_trained"],
    "best epoch": t["best_epoch"],
    "stopped early": t["stopped_early"],
    "learning rate at the end": t["final_learning_rate"],
    "seconds per epoch": t["sec_per_epoch"],
    "training time (min)": t["train_minutes"],
    "peak GPU memory (MB)": t["peak_gpu_memory_mb"],
    "peak host RAM (MB)": t.get("peak_host_ram_mb"),
    "parameters": t["parameters"],
}, name=RUN).to_frame()

## 2. Training curves (T18)

In [ ]:
history = pd.read_csv(run_dir(RUN) / "history.csv")
fig, axes = plt.subplots(1, 3, figsize=(12, 3.2))
viz.plot_history(history, axes[:2], best_epoch=t["best_epoch"])
axes[2].step(history["epoch"] + 1, history["learning_rate"], where="mid", color=viz.INK_2)
axes[2].set_yscale("log")
axes[2].set_xlabel("epoch")
axes[2].set_title("Learning rate")
fig.tight_layout()
viz.savefig(fig, ROOT / "figures" / "R01_training_curves.png")
plt.show()

In [ ]:
best = history.loc[history["val_water_iou"].idxmax()]
last5 = history.tail(5)
pd.Series({
    "best validation IoU": best["val_water_iou"],
    "training IoU at that epoch (augmented crops)": best["water_iou"],
    "validation loss at the best epoch": best["val_loss"],
    "lowest validation loss (epoch %d)" % (history["val_loss"].idxmin() + 1): history["val_loss"].min(),
    "training loss, last 5 epochs (mean)": last5["loss"].mean(),
    "validation loss, last 5 epochs (mean)": last5["val_loss"].mean(),
    "validation IoU, last 5 epochs (mean)": last5["val_water_iou"].mean(),
    "validation IoU: std over the last 10 epochs": history["val_water_iou"].tail(10).std(),
}, name=RUN).to_frame()

## 3. Validation scores, next to the Otsu baseline on the same chips

In [ ]:
otsu = json.loads((run_dir("R00") / "metrics.json").read_text())["val"]
val = metrics["val"]
cols = {"water IoU (pooled)": "iou", "precision": "precision", "recall": "recall", "F1": "f1", "mean per-chip IoU": "mean_chip_iou"}
pd.DataFrame({"Otsu on VH (R00)": {k: otsu[v] for k, v in cols.items()},
              "Default U-Net (R01)": {k: val[v] for k, v in cols.items()}}).T

In [ ]:
chips = pd.read_csv(run_dir(RUN) / "per_chip.csv")
chips = chips[chips["split"] == "val"].copy()
otsu_chips = pd.read_csv(run_dir("R00") / "per_chip.csv")
otsu_chips = otsu_chips[otsu_chips["split"] == "val"].set_index("chip")
chips["otsu_iou"] = chips["chip"].map(otsu_chips["iou"])

by_event = chips.groupby("event").agg(chips=("chip", "size"), tp=("tp", "sum"), fp=("fp", "sum"), fn=("fn", "sum"),
                                      water_px=("n_water", "sum"))
by_event["U-Net IoU"] = by_event["tp"] / (by_event["tp"] + by_event["fp"] + by_event["fn"])
o = otsu_chips.groupby("event")[["tp", "fp", "fn"]].sum()
by_event["Otsu IoU"] = o["tp"] / (o["tp"] + o["fp"] + o["fn"])
by_event["precision"] = by_event["tp"] / (by_event["tp"] + by_event["fp"])
by_event["recall"] = by_event["tp"] / (by_event["tp"] + by_event["fn"])
by_event[["chips", "water_px", "Otsu IoU", "U-Net IoU", "precision", "recall"]].sort_values("U-Net IoU")

In [ ]:
with_water = chips[chips["n_water"] > 0]
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
axes[0].scatter(with_water["water_frac"].clip(lower=1e-4), with_water["iou"], s=22, color=viz.BLUE, edgecolor="white", linewidth=0.6)
axes[0].set_xscale("log")
axes[0].set_xlabel("water share of the chip (log scale)")
axes[0].set_ylabel("per-chip IoU, default U-Net")
axes[0].set_ylim(-0.03, 1.03)
axes[0].set_title("Chips with little water score worst")

axes[1].scatter(with_water["otsu_iou"], with_water["iou"], s=22, color=viz.BLUE, edgecolor="white", linewidth=0.6)
axes[1].plot([0, 1], [0, 1], color=viz.AXIS, linewidth=1)
axes[1].set_xlabel("per-chip IoU, Otsu on VH")
axes[1].set_ylabel("per-chip IoU, default U-Net")
axes[1].set_xlim(-0.03, 1.03)
axes[1].set_ylim(-0.03, 1.03)
axes[1].set_title("Chip by chip against the baseline (above the line = U-Net better)")
fig.tight_layout()
viz.savefig(fig, ROOT / "figures" / "R01_val_per_chip.png")
plt.show()

no_water = chips[chips["n_water"] == 0]
print(f"validation chips with water: {len(with_water)} | U-Net better than Otsu on {(with_water['iou'] > with_water['otsu_iou']).sum()} of them")
print(f"validation chips without water: {len(no_water)} | U-Net predicts no water at all on {(no_water['fp'] == 0).sum()}, "
      f"median false-alarm pixels on the others: {no_water.loc[no_water['fp'] > 0, 'fp'].median()}")

## 4. Predictions on validation chips (T18)

Six validation chips chosen by a fixed rule from the chips with at least 1% water: the two with the lowest IoU, the two around the median, and the two with the highest.
Columns: VH, the hand label, and the prediction coloured by outcome.

In [ ]:
from src.data import load_split
from src.evaluate import load_run_model, predict_proba
from src.pipeline import load_arrays
from src.train import setup_runtime

setup_runtime(cfg["mixed_precision"])
model = load_run_model(RUN)
val_data = load_split("val")
X_val, _ = load_arrays("val", cfg["channels"])
probs = predict_proba(model, X_val, cfg["eval_batch_size"])
pred = probs >= cfg["threshold"]

ranked = chips[chips["water_frac"] >= 0.01].sort_values("iou").reset_index(drop=True)
mid = len(ranked) // 2
picks = pd.concat([ranked.iloc[:2], ranked.iloc[mid - 1:mid + 1], ranked.iloc[-2:]])
index = {n: i for i, n in enumerate(val_data.names)}

fig, axes = plt.subplots(3, 6, figsize=(11.5, 6.3))
for col, (_, row) in enumerate(picks.iterrows()):
    i = index[row["chip"]]
    viz.show_db(axes[0, col], val_data.images[i, 1], "VH", title=row["chip"].replace("_", " "))
    viz.show_label(axes[1, col], val_data.labels[i], title=f"{row['water_frac']:.0%} water")
    viz.show_errors(axes[2, col], pred[i], val_data.labels[i], title=f"IoU {row['iou']:.2f}  (Otsu {row['otsu_iou']:.2f})")
for r, text in enumerate(("VH (dB)", "hand label", "prediction vs label")):
    axes[r, 0].set_ylabel(text, fontsize=9, color=viz.INK_2)
fig.legend(handles=viz.error_legend_handles(), loc="lower center", ncol=5, bbox_to_anchor=(0.5, -0.03))
fig.suptitle("lowest IoU  ←                                      median                                      →  highest IoU",
             fontsize=9, color=viz.MUTED, y=0.995)
fig.tight_layout(h_pad=0.5, w_pad=0.4)
viz.savefig(fig, ROOT / "figures" / "R01_val_predictions.png")
plt.show()

## 5. Is 0.5 a sensible threshold? (validation only)

The plan fixes the decision threshold at 0.5. This check only asks whether that is unreasonable for this model; it does not tune anything.

In [ ]:
labels = val_data.labels
valid, truth = labels != -1, labels == 1
rows = []
for thr in (0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9):
    p = probs >= thr
    tp, fp, fn = int((p & truth).sum()), int((p & valid & ~truth).sum()), int((~p & truth).sum())
    rows.append({"threshold": thr, "IoU": tp / (tp + fp + fn), "precision": tp / (tp + fp), "recall": tp / (tp + fn)})
thr_table = pd.DataFrame(rows).set_index("threshold")
thr_table.to_csv(run_dir(RUN) / "val_threshold_check.csv")
confident = float(((probs < 0.05) | (probs > 0.95))[valid].mean())
print(f"share of validation pixels with a probability below 0.05 or above 0.95: {confident:.1%}")
thr_table.T